# 02 — Corpus Cleaning and Normalization

Crea una copia limpia, normaliza texto y fuentes, y prepara afiliaciones con trazabilidad.

> Notebook reproducible del Paper Semantic Bibliometric Toolkit v1.0.0. No elimina documentos ni convierte resultados exploratorios en conclusiones sin revisión manual.

## 1. Cargar corpus

In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
current = Path.cwd().resolve()
for candidate in [current] + list(current.parents):
    if (candidate / "config.json").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.json.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core import (
    find_project_root, read_json, load_config, ensure_directories,
    load_scopus_csv, load_clean_corpus, normalize_spacing,
    normalize_identifier, normalize_title, normalize_keyword,
    split_semicolon, frequency_table, save_csv, save_json,
    markdown_table, combined_text
)

CONFIG = load_config(PROJECT_ROOT)
ensure_directories(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
THESAURUS_DIR = PROJECT_ROOT / "thesaurus"
VALIDATION_DIR = PROJECT_ROOT / "validation"

print("Project root:", PROJECT_ROOT)
print("Toolkit version:", CONFIG.get("toolkit_version"))
print("Configured corpus:", PROJECT_ROOT / CONFIG["input_csv"])

raw_df = load_scopus_csv(PROJECT_ROOT, CONFIG)
clean_df = raw_df.copy()


## 2. Normalización conservadora de texto

In [ ]:

text_columns = [column for column in clean_df.columns if clean_df[column].dtype == "object"]
for column in text_columns:
    clean_df[column] = clean_df[column].map(normalize_spacing)
clean_df["EID normalized"] = clean_df["EID"].map(normalize_identifier)
clean_df["DOI normalized"] = clean_df["DOI"].map(normalize_identifier)
clean_df["Title normalized"] = clean_df["Title"].map(normalize_title)
print("Text columns normalized:", len(text_columns))


## 3. Normalización de fuentes y candidatos similares

In [ ]:

from difflib import SequenceMatcher
source_map = read_json(PROJECT_ROOT / "config" / "source_replacements.json")
clean_df["Source title normalized"] = clean_df["Source title"].map(lambda value: source_map.get(normalize_spacing(value), normalize_spacing(value)))
audit = clean_df[["Source title", "Source title normalized"]].drop_duplicates().sort_values(["Source title normalized", "Source title"])
applied = audit[audit["Source title"] != audit["Source title normalized"]].copy()
save_csv(audit, OUTPUT_DIR / "02_source_normalization_audit.csv")
save_csv(applied, OUTPUT_DIR / "02_source_replacements_applied.csv")

sources = sorted(set(clean_df["Source title normalized"]) - {""})
candidates = []
threshold = float(CONFIG.get("source_similarity_threshold", 0.82))
for index, left in enumerate(sources):
    for right in sources[index + 1:]:
        ratio = SequenceMatcher(None, normalize_title(left), normalize_title(right)).ratio()
        if ratio >= threshold:
            candidates.append({"Source A": left, "Source B": right, "Similarity": round(ratio, 3), "Decision": "review"})
source_candidates = pd.DataFrame(candidates) if candidates else pd.DataFrame(columns=["Source A", "Source B", "Similarity", "Decision"])
save_csv(source_candidates, OUTPUT_DIR / "02_source_variant_candidates.csv")
display(applied)
display(source_candidates.head(30))


## 4. Afiliaciones e instituciones candidatas

In [ ]:

from parsing import parse_affiliations, extract_country_from_affiliation, extract_institution_candidate
rows = []
for _, record in clean_df.iterrows():
    for affiliation in parse_affiliations(record.get("Affiliations", "")):
        rows.append({
            "EID": record.get("EID", ""), "Title": record.get("Title", ""), "Year": record.get("Year", ""),
            "Affiliation original": affiliation,
            "Institution candidate": extract_institution_candidate(affiliation),
            "Country candidate": extract_country_from_affiliation(affiliation)
        })
affiliations = pd.DataFrame(rows)
save_csv(affiliations, OUTPUT_DIR / "02_affiliations_exploded.csv")
institution_frequency = affiliations.drop_duplicates(["EID", "Institution candidate"]).groupby("Institution candidate")["EID"].nunique().sort_values(ascending=False).rename("Documents").reset_index()
country_frequency = affiliations[affiliations["Country candidate"].ne("")].drop_duplicates(["EID", "Country candidate"]).groupby("Country candidate")["EID"].nunique().sort_values(ascending=False).rename("Documents").reset_index()
save_csv(institution_frequency, OUTPUT_DIR / "02_institution_candidate_frequency.csv")
save_csv(country_frequency, OUTPUT_DIR / "02_country_candidate_frequency.csv")
display(institution_frequency.head(20))


## 5. Auditoría, exportación y metodología

In [ ]:

changes = []
for column in text_columns:
    original = raw_df[column].fillna("").astype(str)
    cleaned = clean_df[column]
    changed = int((original != cleaned).sum())
    changes.append({"Column": column, "Changed rows": changed, "Percentage changed": round(changed / len(clean_df) * 100, 2)})
cleaning_audit = pd.DataFrame(changes).sort_values("Changed rows", ascending=False)
save_csv(cleaning_audit, OUTPUT_DIR / "02_cleaning_audit.csv")
save_csv(clean_df, OUTPUT_DIR / "corpus_cleaned.csv")
summary = {
    "Documents": len(clean_df), "Original columns": len(raw_df.columns), "Cleaned columns": len(clean_df.columns),
    "Text columns normalized": len(text_columns), "Explicit source replacements": len(applied),
    "Source variant candidates": len(source_candidates), "Expanded affiliations": len(affiliations)
}
save_json(summary, OUTPUT_DIR / "02_cleaning_summary.json")
text = (
    "The original Scopus export containing {} records was preserved unchanged. A cleaned copy was generated by "
    "normalizing Unicode characters, typographic punctuation, line breaks, and repeated whitespace across {} textual fields. "
    "No documents were deleted, and all source-title harmonization rules were audited."
).format(len(raw_df), len(text_columns))
(REPORTS_DIR / "02_cleaning_methodology.txt").write_text(text, encoding="utf-8")
print(text)
